---
title: Creating a Schema
---

import CodeBlock from "@theme/CodeBlock";
import CodeOutputBlock from "@theme/CodeBlock";
import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';

# Creating a schema

In Infrahub, schemas play a crucial role in defining the structure and relationships of your data.
This guide will walk you through the process of creating a new schema file, we will do this using the following steps:

1. Adding nodes and attributes
2. Adding relationships to the nodes
3. Abstracting nodes into generics
4. Improving our schema by modifying attributes and adding new attributes

In this guide we will create a schema for a network device and its interfaces.

By no mean this is meant to be a complete schema for a network device, there is far more complexity that goes into modeling a schema for a network device, it is just used as an example to guide you through the process.

For a more detailed explanation of the different Attributes and Relations, you can check out our [Schema Topic](/topics/schema).

Throughout this guide we will load different schemas into Infrahub into different branches. The fact that multiple schemas can exist simultaneously is a core feature of Infrahub. It is a recommendation and best practice to always use a branch to make changes to the schema.

:::note

To help with the development process of a schema definition file, you can leverage [schema validation](/reference/schema-validation) within your editor.

:::

## 1. Adding nodes and attributes

Create a file named `schema_guide.yml` on your computer. The place where you create the file on the file system is not that important, as long as you know the path to the file. For this guide we will be storing the schema file in the `/tmp` directory.

In the file we will be creating 2 kinds of nodes in the `Network` namespace:

- `Device`: the network device we want to model
- `Interface`: the network interface we want to model

The `NetworkDevice` node will have the following attributes:

- `hostname`: the `hostname` of the device, needs to be unique and is a required attribute
- `model` (Text): the model of the device, which is a required attribute

The `NetworkInterface` node will have the following attributes:

- `name` (Text): the name of the interface, which is a required attribute
- `description` (Text): a description for the interface, which is a required attribute

:::note

We define a `human_friendly_id` on the `hostname` attribute of the `NetworkDevice`. This way we can use the `hostname` as an alternative for the `hfid` in the queries and mutations in this guide.

:::

In [1]:
%%writefile /tmp/schema_guide.yml
---
version: "1.0"
nodes:
  - name: Device
    namespace: Network
    human_friendly_id: ['hostname__value']
    attributes:
      - name: hostname
        kind: Text
        unique: true
      - name: model
        kind: Text
  - name: Interface
    namespace: Network
    attributes:
      - name: name
        kind: Text
      - name: description
        kind: Text
        optional: true

Overwriting /tmp/schema_guide.yml


Create a branch `network-device-schema` in Infrahub.


In [2]:
%set_env INFRAHUB_API_TOKEN=06438eb2-8019-4776-878c-0941b1f1d1ec
!infrahubctl branch create "network-device-schema"

env: INFRAHUB_API_TOKEN=06438eb2-8019-4776-878c-0941b1f1d1ec


Branch 'network-device-schema' created successfully 
(180bf4f0-931e-a773-3683-c51999ac33c0).


Load the schema into Infrahub in the `network-device-schema` branch

In [3]:
!infrahubctl schema load --branch network-device-schema /tmp/schema_guide.yml

 schema '/tmp/schema_guide.yml' loaded successfully
 1 schema processed in 6.839 seconds.


We can inspect the schema in the [Web UI](http://localhost:8000/schema?branch=network-device-schema) (Unified Storage > Schema) as shown below.

![schema page screenshot](../media/guides/create_schema_1.png)

We'll create a device and an interface according to the schema by using the web interface, GraphQL Query or cURL.

<Tabs>
<TabItem value="graphql" label="Via the GraphQL Interface" default>

Open the GraphQL sandbox (bottom left of the web interface) and make a query with the following:

```graphql
mutation {
  NetworkDeviceCreate(data: {hostname: {value: "atl1-edge1"}, model: {value: "Cisco ASR1002-HX"}}) {
        ok
        object {
          id
        }
  }
  NetworkInterfaceCreate(data: {name: {value: "Ethernet1"}, description: {value: "WAN interface"}}) {
        ok
        object {
          id
        }
  }
}
```

![schema page screenshot](../media/guides/create_schema_graphql_1.png)

 </TabItem>

  <TabItem value="web" label="Via the Web Interface">

1. Login to Infrahub's web interface.
2. Click on **Objects > Device** in the left side menu.
3. Click on 'Add Device'
4. Create a new device with 'atl1-edge1' as Hostname and 'Cisco ASR1002-HX' as Model and click save.
5. Under Interface, create a new interface with 'Ethernet1' as Name and 'WAN interface' as description.

  </TabItem>

 <TabItem value="shell" label="Using cURL">

Here is an example of using `cURL` to make the query. Make sure to replace the `X-INFRAHUB-KEY` and the IP address with your actual details. Please also make sure to include the name of the branch in the URL. If you want to learn more about GraphQL, you can find more information [here](https://docs.infrahub.app/topics/graphql).

```shell
curl -X POST http://localhost:8000/graphql/network-device-schema \
  -H "Content-Type: application/json" \
  -H "X-INFRAHUB-KEY: 1802eed5-eeb7-cc45-2e4d-c51de9d66cba" \
  -d '{"query": "mutation { NetworkDeviceCreate(data: {hostname: {value: \"atl1-edge1\"}, model: {value: \"Cisco ASR1002-HX\"}}) { ok object { id } } NetworkInterfaceCreate(data: {name: {value: \"Ethernet1\"}, description: {value: \"WAN interface\"}}) { ok object { id } } }"}'
```

 </TabItem>
</Tabs>